In [148]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [149]:
from index.inverted_index import Index
from index.documents import load_dataset, tokenizer

msmarco = load_dataset("msmarco-passage/dev/judged")

In [150]:
import locale, sys
print(sys.flags.utf8_mode)                 # should be 1
print(locale.getpreferredencoding(False))  # should be UTF-8

1
UTF-8


In [151]:
NUM_DOCS = 1000
K = 10

In [152]:
msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=NUM_DOCS)

{'the': [(1, 6),
  (2, 1),
  (3, 5),
  (4, 8),
  (5, 4),
  (6, 4),
  (7, 5),
  (8, 14),
  (9, 4),
  (10, 6),
  (12, 2),
  (13, 2),
  (14, 4),
  (15, 4),
  (16, 6),
  (17, 3),
  (18, 5),
  (19, 3),
  (20, 8),
  (21, 6),
  (22, 2),
  (23, 2),
  (24, 3),
  (25, 3),
  (27, 5),
  (28, 6),
  (29, 1),
  (30, 5),
  (31, 3),
  (32, 5),
  (33, 5),
  (34, 1),
  (35, 8),
  (36, 6),
  (37, 8),
  (38, 7),
  (39, 2),
  (41, 1),
  (42, 1),
  (43, 2),
  (45, 5),
  (46, 5),
  (47, 4),
  (48, 5),
  (49, 1),
  (50, 2),
  (51, 1),
  (52, 6),
  (53, 1),
  (54, 4),
  (55, 1),
  (56, 2),
  (57, 5),
  (58, 1),
  (59, 4),
  (60, 1),
  (61, 1),
  (62, 5),
  (63, 17),
  (64, 6),
  (65, 3),
  (66, 3),
  (67, 1),
  (68, 5),
  (69, 6),
  (70, 9),
  (71, 5),
  (72, 2),
  (73, 8),
  (74, 7),
  (75, 1),
  (76, 3),
  (77, 8),
  (78, 6),
  (79, 5),
  (80, 2),
  (81, 4),
  (82, 3),
  (83, 7),
  (84, 8),
  (85, 8),
  (86, 10),
  (87, 6),
  (88, 3),
  (89, 2),
  (91, 2),
  (93, 1),
  (94, 1),
  (97, 4),
  (99, 1),
  (100, 3

## Sparse tf_idf Matrix

In [153]:
from scipy.sparse import csr_matrix
import numpy as np
import pandas as pd

def create_sparse_tf_idf_matrix(index, dtype=np.float32):
    terms = list(index.get_terms())
    term_to_col = {t:i for i,t in enumerate(terms)}
    doc_ids = index.get_doc_ids()
    doc_to_row = {did:i for i,did in enumerate(doc_ids)}

    rows, cols, data = [], [], []
    for term, posting in index.index.items():
        idf = index.get_inverse_document_frequency(term)
        col = term_to_col[term]
        for doc_id, tf in posting:
            rows.append(doc_to_row[doc_id])
            cols.append(col)
            data.append(np.log10(1+tf) * idf)

    data = np.array(data, dtype=dtype)
    mat = csr_matrix((data, (rows, cols)), shape=(len(doc_ids), len(terms)), dtype=dtype)
    return mat, doc_ids, terms

In [154]:
mat, doc_ids, terms = create_sparse_tf_idf_matrix(msmarco_index)
df = pd.DataFrame.sparse.from_spmatrix(mat, index=doc_ids, columns=terms)

In [155]:
# term_index = {term: idx for idx, term in enumerate(terms)}
# # Transform each document into a set of termIDs

# doc_set = {doc_id: set(term_index[term] for term in msmarco.docs_iter()[1] f ) for doc_id in doc_ids}

In [156]:
doc_id_term_id = msmarco_index.compute_docId_termId(msmarco, tokenizer, max_num_docs=NUM_DOCS)

{1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 0, 11, 2, 0, 12, 13, 14, 5, 15, 7, 0, 16, 17, 18, 19, 0, 20, 21, 2, 0, 22, 23, 24, 25, 26, 27, 28, 11, 29, 30, 31, 2, 32, 2, 33, 34, 35], 2: [0, 12, 13, 24, 36, 22, 37, 38, 39, 40, 41, 10, 42, 43, 44, 36, 45, 2, 46, 47, 2, 22, 48, 49, 10, 50, 40, 51, 52, 53, 24, 54], 3: [55, 52, 0, 12, 13, 0, 12, 13, 0, 12, 13, 7, 10, 56, 57, 58, 40, 22, 37, 59, 0, 11, 2, 60, 13, 61, 62, 63, 0, 42, 62, 58, 64, 65, 66, 67, 60, 68, 69, 70, 71], 4: [0, 12, 13, 7, 0, 72, 73, 74, 13, 75, 76, 42, 43, 44, 10, 77, 0, 78, 22, 37, 64, 79, 80, 10, 0, 81, 2, 0, 13, 82, 83, 84, 85, 86, 0, 87, 2, 0, 88, 89, 90, 2, 25, 86, 0, 91, 2, 92, 93, 94, 95], 5: [96, 2, 97, 98, 14, 99, 14, 100, 101, 0, 78, 102, 12, 13, 40, 103, 104, 105, 52, 0, 106, 2, 53, 24, 107, 108, 109, 110, 111, 0, 106, 2, 53, 24, 107, 108, 24, 0, 112, 113, 114], 6: [0, 12, 13, 60, 115, 116, 117, 118, 0, 78, 22, 37, 119, 74, 120, 66, 22, 121, 122, 123, 124, 0, 113, 125, 126, 52, 127, 128, 129, 130, 64, 7, 131, 132, 

In [157]:
# Reorder the collection according to the size of document set (reverse order)
# doc_ids = sorted(doc_id_term_id, key=lambda x: len(doc_id_term_id.get(x)), reverse=True)
doc_id_term_id = {
    doc_id: doc_id_term_id[doc_id] 
    for doc_id in sorted(doc_id_term_id, key=lambda x: len(doc_id_term_id.get(x)), reverse=True)
}
# Scan linearly the collection of document to clustering them using the Jaccard distance = 1 - JaccardSimilarity

print(doc_id_term_id)

{63: [398, 1007, 989, 205, 1071, 26, 74, 1072, 205, 1073, 833, 2, 0, 986, 66, 1074, 1075, 0, 1076, 1018, 1077, 340, 0, 986, 1018, 26, 74, 1078, 895, 66, 26, 454, 132, 0, 986, 64, 458, 254, 0, 1018, 1077, 10, 0, 1079, 1080, 1081, 64, 930, 1082, 1083, 24, 1084, 1085, 703, 158, 1071, 0, 1018, 1077, 344, 1086, 155, 1087, 60, 1088, 1018, 10, 1089, 132, 0, 986, 1081, 1090, 1091, 1092, 986, 447, 24, 1088, 989, 205, 1093, 2, 0, 1094, 989, 1095, 24, 0, 317, 2, 1096, 410, 132, 0, 986, 1097, 0, 986, 1098, 36, 1099, 10, 1100, 458, 254, 0, 1018, 1077, 10, 0, 1079, 1080, 1081, 64, 930, 1082, 1083, 24, 1084, 1101, 132, 703, 158, 1071, 0, 1018, 1077, 344, 1086, 155, 1087, 60, 1088, 1018, 10, 1089, 132, 0, 986, 1081, 1090, 1091, 1092, 986, 447, 24, 1088, 989, 205, 1093, 2, 0, 986], 692: [132, 60, 2245, 6725, 6727, 6739, 702, 5005, 65, 14, 6725, 344, 6740, 195, 1158, 323, 6725, 2031, 271, 6734, 205, 2187, 6725, 24, 6725, 2031, 180, 6741, 205, 6724, 6725, 590, 5005, 6725, 2031, 271, 1088, 6735, 1113, 0, 

In [158]:
# Stream_cluster(SortedCollection, Radius)
# C = ∅
# for each d in SortedCollection
# Dist_c = Min (JaccardDistance(c, d), for each medoid c in C)
# if (Dist_c < Radius) then
# add d to cluster c
# else
# make d a new medoid, and add it to C: C = C ∪ d
# return C

In [159]:
from sklearn.metrics import jaccard_score

# def jaccard_distance(x, y):
#     intersection = np.sum(np.minimum(x, y))
#     union = np.sum(np.maximum(x, y))
#     return 1 - intersection / union if union > 0 else 0

# def stream_cluster(SortedCollection, Radius, k):
#     C = []
#     Clusters = {}

#     for d in SortedCollection:
#             if C:
#                 # Calcoli la lista delle distanze una volta sola
#                 distances = [1 - jaccard_score(list(c), list(d), average='binary') for c in C]
                
#                 # Estrai indice e valore minimo senza ricalcolare
#                 cluster = np.argmin(distances)
#                 dist_c = distances[cluster]
#             else:
#                 cluster, dist_c = 1, 0

#     if dist_c < Radius:
#         Clusters[cluster].append(d)
#     else:
#         # make d a new medoid, and add it to C
#         C.append(d)
#     return C

from collections import defaultdict
from sklearn.metrics import jaccard_score
import numpy as np

def stream_cluster(SortedCollection, Radius, k):
    C = []                      # Conterrà le liste di term_id dei medoidi
    Clusters = defaultdict(list) # Conterrà gli ID (chiavi) dei documenti suddivisi per cluster

    # Iteriamo usando .items() per avere sia l'ID del documento che la sua lista di termini
    for doc_id, doc_terms in SortedCollection.items():
        
        set_d = set(doc_terms)
        # Se C è vuoto, il primo documento diventa il primo medoide
        if not C:
            Clusters[0].append(doc_id) # Salva l'ID nel cluster 0
            C.append(doc_id)
            continue

        # Calcolo delle distanze usando doc_terms
        distances = []
        for c in C:
            # print(SortedCollection)
            centroid_set = set(SortedCollection.get(c))
            # set_c = set(c) # Recuperiamo i termini del medoide tramite il suo ID 'c'
            
            # Formula di Jaccard: 1 - (Intersezione / Unione)
            intersection = len(centroid_set.intersection(set_d))
            union = len(centroid_set.union(set_d))
            
            jaccard_dist = 1 - (intersection / union) if union > 0 else 1
            distances.append(jaccard_dist)
        
        print(distances)
        cluster = np.argmin(distances)
        dist_c = distances[cluster]

        if dist_c < Radius:
            # Assegna l'ID del documento al cluster esistente
            Clusters[cluster].append(doc_id)
        else:
            # Crea un nuovo medoide con i termini del documento corrente
            C.append(doc_id)
            nuovo_indice_cluster = len(C) - 1
            Clusters[nuovo_indice_cluster].append(doc_id)
            
    return C, Clusters

In [160]:
C, Clusters = stream_cluster(doc_id_term_id, 0.95, 4)
print(Clusters)

[0.8976377952755905]
[0.956140350877193]
[0.9140625, 0.9612403100775194]
[0.9099099099099099, 0.9259259259259259]
[0.9428571428571428, 0.9328358208955224]
[0.9504950495049505, 0.9479166666666666]
[0.8983050847457628, 0.9130434782608696]
[0.9572649572649573, 0.9646017699115044]
[0.9259259259259259, 0.9223300970873787, 0.9636363636363636]
[0.9166666666666666, 0.9495798319327731, 0.967741935483871]
[0.9105691056910569, 0.9512195121951219, 0.9606299212598425]
[0.8617021276595744, 0.9591836734693877, 0.95]
[0.8910891089108911, 0.9509803921568627, 0.9622641509433962]
[0.9145299145299145, 0.9391304347826087, 0.9130434782608696]
[0.9153846153846154, 0.9457364341085271, 0.9626865671641791]
[0.9243697478991597, 0.9396551724137931, 0.95]
[0.884297520661157, 0.9516129032258065, 0.9609375]
[0.9147286821705426, 0.937007874015748, 0.9545454545454546]
[0.9262295081967213, 0.9043478260869565, 0.9338842975206612]
[0.9196428571428571, 0.9454545454545454, 0.9469026548672567]
[0.9069767441860466, 0.9538461

## Kmeans + TCP

In [161]:
# from sklearn.cluster import KMeans, DBSCAN
# from sklearn.preprocessing import normalize

# def jaccard_distance(x, y):
#     intersection = np.sum(np.minimum(x, y))
#     union = np.sum(np.maximum(x, y))
#     return 1 - intersection / union if union > 0 else 0

# clustering = DBSCAN(metric=jaccard_distance, eps=0.5, min_samples=2)

# # norm_df = normalize(df, norm="l2") #! cosa fa , è fondamentele se non la metti clusteing fa schifo 

# # kmeans = KMeans(n_clusters=K, random_state=0) #! use jaccard distance instead of euclidean

# clusters = clustering.fit(doc_id_term_id.values())
# print("Cluster centers:", clustering.cluster_centers_)

# document_clusters = clustering.labels_
# print("Document clusters:", document_clusters)

## Medoids

In [162]:
# medoids = []

# for i in range(kmeans.n_clusters):
#     cluster_docs = np.flatnonzero(document_clusters == i)
#     cluster_vectors = norm_df[cluster_docs]
#     medoid_idx = np.argmin(np.linalg.norm(cluster_vectors - kmeans.cluster_centers_[i], axis=1))
#     medoids.append(cluster_docs[medoid_idx])



# medoid_vectors = norm_df[medoids]
# nonzero_cols = medoid_vectors.getnnz(axis=0) > 0
# medoid_vectors = medoid_vectors[:, nonzero_cols].toarray()


In [163]:
from sklearn.metrics import pairwise_distances
import numpy as np

def calculate_distance_matrix(medoid_vectors, method="euclidean"):
    
    dist = np.zeros((len(medoid_vectors), len(medoid_vectors)))

    for i, v1 in enumerate(medoid_vectors):
        set_v1 = set(v1)

        for j, v2 in enumerate(medoid_vectors):

            if(i == j):
                continue
            
            set_v2 = set(v2)            
            # Formula di Jaccard: 1 - (Intersezione / Unione)
            intersection = len(set_v1.intersection(set_v2))
            union = len(set_v1.union(set_v2))
            
            jaccard_dist = 1 - (intersection / union) if union > 0 else 1
            dist[i][j] = jaccard_dist
        
    return dist
    
    # if method == "euclidean":
    #     return pairwise_distances(medoid_vectors, metric="euclidean")
    # elif method == "cosine":
    #     return pairwise_distances(medoid_vectors, metric="cosine")
    # elif method == "jaccard":
    #     # Jaccard in sklearn works on boolean/binary features
    #     return pairwise_distances(medoid_vectors, metric="jaccard")
    # else:
    #     raise ValueError(f"Unknown method: {method}")

In [164]:
medoid_vectors = [doc_id_term_id.get(c) for c in C]

In [165]:
dist_matrix = calculate_distance_matrix(medoid_vectors, method="jaccard")

In [166]:
import networkx as nx
tsp = nx.approximation.traveling_salesman_problem

tsp_graph = nx.from_numpy_array(dist_matrix)
tsp_path = tsp(tsp_graph, cycle=False)
print("TSP path (medoid indices):", tsp_path)

TSP path (medoid indices): [14, 15, 17, 9, 6, 3, 0, 18, 10, 2, 20, 12, 24, 1, 4, 13, 25, 11, 8, 5, 23, 7, 22, 26, 21, 19, 16]


In [167]:
# def reorder_documents_by_tsp(document_clusters, medoids, tsp_path):
#     """
#     Reorder documents by cluster order induced by TSP over medoids.
#     Inside each cluster, keep documents in their original order (row order).

#     Args:
#         document_clusters: array-like of shape (n_docs,), cluster label per doc.
#         medoids: list of medoid doc indices (not required for ordering here, kept for API compatibility).
#         tsp_path: list of cluster ids in the order returned by TSP.

#     Returns:
#         ordered_doc_ids: list of document row indices in final order.
#         cluster_to_docs: dict {cluster_id: [doc_ids in original order]}.
#     """
#     document_clusters = np.asarray(document_clusters)

#     # Build cluster -> docs preserving natural/original order
#     cluster_to_docs = {
#         c: np.flatnonzero(document_clusters == c).tolist()
#         for c in range(len(medoids))
#     }

#     # Concatenate clusters following TSP order
#     ordered_doc_ids = []
#     for c in tsp_path:
#         ordered_doc_ids.extend(cluster_to_docs.get(c, []))

#     return ordered_doc_ids, cluster_to_docs

In [168]:
# ordered_doc_ids, cluster_docs = reorder_documents_by_tsp(Clusters, C, tsp_path)

In [169]:
def reorder_documents_by_tsp_new(tsp_path, cluster, C):
    tot = []
    for i in tsp_path:
        docs = cluster.get(i)
        for d in docs:  
            tot.append(d)
    return tot

reord_docids = reorder_documents_by_tsp_new(tsp_path,Clusters,C)
print(min(reord_docids))

1


In [173]:
from itertools import islice

docs = list(islice(msmarco.docs_iter(), NUM_DOCS))

class OrderedDataset:
    def __init__(self, docs, order):
        self.docs = docs
        self.order = order
    def docs_iter(self):
        for i in self.order:
            yield self.docs[i-1]

ordered_dataset = OrderedDataset(docs, reord_docids)
reordered_inverted_index = Index()
reordered_inverted_index.build_index(
    ordered_dataset,
    tokenizer,
    max_num_docs=-1, #! -1 since reordered docs are already limited to 10001 in the original index
    d_gap=True
)

stat_r = reordered_inverted_index.get_approx_size()

msmarco_index = Index()
msmarco_index.build_index(msmarco, tokenizer, max_num_docs=1000, d_gap=True)

stat_m = msmarco_index.get_approx_size()
print(f"Original index size: {stat_m} bytes")
print(f"Reordered index size: {stat_r} bytes")

ind_1 = reordered_inverted_index.index
ind_2 = msmarco_index.index

print("Msmarco index size:", len(msmarco_index.index))


=== Index Size Breakdown ===
Doc ID bits:  1,202,464
TF bits:      1,202,464
Total bits:   2,404,928
Total bytes:  300,616.00

=== Index Size Breakdown ===
Doc ID bits:  1,202,464
TF bits:      1,202,464
Total bits:   2,404,928
Total bytes:  300,616.00
Original index size: {'term_bytes': 0, 'doc_id_bits': 1202464, 'tf_bits': 1202464, 'total_bits': 2404928, 'total_bytes': 300616.0} bytes
Reordered index size: {'term_bytes': 0, 'doc_id_bits': 1202464, 'tf_bits': 1202464, 'total_bits': 2404928, 'total_bytes': 300616.0} bytes
Msmarco index size: 8495


In [174]:
reordered_inverted_index.compress_index("gamma")
reordered_inverted_index.save_index(file_path="reord.bin")
msmarco_index.compress_index("gamma")
msmarco_index.save_index(file_path="not_reord.bin")